In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing._encoders import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv("../data/processed/featured.csv")

# 1 Defining the X and Y

In [ ]:
X = df.drop("Time_taken(min)",axis=1)
y = df["Time_taken(min)"]

# 2 Split 80/20

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 3 Identifying numerical and categorical features

In [ ]:
numerical_features = X_train.select_dtypes(include=["Int64","Float64"]).columns
categorical_features = X_train.select_dtypes(exclude=["Int64","Float64"]).columns

# 4 Build the preprocessing pipeline

In [ ]:
numerical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_pipeline = Pipeline([
    ("impute",SimpleImputer(strategy="most_frequent")),
    ("scaler",OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ("num",numerical_pipeline,numerical_features),
    ("cat",categorical_pipeline,categorical_features)
])

# 5 Connect the preprocessor to the models

In [ ]:
linear_regression_model = Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",LinearRegression())
])

decision_tree_model = Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",DecisionTreeRegressor())
])

random_forest_model = Pipeline([
    ("prepprocessor",preprocessor),
    ("regressor",RandomForestRegressor())
])

gradient_boosting_model = Pipeline([
    ("prepprocessor",preprocessor),
    ("regressor",GradientBoostingRegressor())
])

models = {
    "Linear Regression" : linear_regression_model,
    "Decision Tree Regressor" : decision_tree_model,
    "Random Forest" : random_forest_model,
    "Gradient Boosting" : gradient_boosting_model
}

# 6 Evaluate the models

In [ ]:
testing_df = pd.DataFrame(columns=["MAE","MSE","RMSE","R2(%)"],index=["Linear Regression","Decision Tree Regressor","Random Forest","Gradient Boosting"])
training_df = pd.DataFrame(columns=["MAE","MSE","RMSE","R2(%)"],index=["Linear Regression","Decision Tree Regressor","Random Forest","Gradient Boosting"])
x_validation_df = pd.DataFrame(columns=["Mean MAE","MAE_STD","Mean MSE","MSE_STD","Mean RMSE","RMSE_STD","Mean R2","R2_STD"],index=["Linear Regression","Decision Tree Regressor","Random Forest","Gradient Boosting"])

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

for name, model in models.items():
    for metric, scoring_name in scoring.items():
        scores = cross_val_score(
            model, X_train, y_train,
            cv=5,
            scoring=scoring_name
        )

        if scoring_name.startswith("neg_"):
            scores = -scores

        x_validation_df.loc[name, f"Mean {metric}"] = scores.mean()
        x_validation_df.loc[name, f"{metric}_STD"] = scores.std()

for name,model in models.items():
    model.fit(X_train,y_train)

    predictions = model.predict(X_test)
    test_evaluations = [
        mean_absolute_error(y_test,predictions),
        mean_squared_error(y_test,predictions),
        np.sqrt(mean_squared_error(y_test,predictions)),
        round(r2_score(y_test,predictions)*100,2)
    ]
    testing_df.loc[name] = test_evaluations

    predictions = model.predict(X_train)
    train_evaluations = [
        mean_absolute_error(y_train,predictions),
        mean_squared_error(y_train,predictions),
        np.sqrt(mean_squared_error(y_train,predictions)),
        round(r2_score(y_train,predictions)*100,2)
    ]
    training_df.loc[name] = train_evaluations

print("Cross Validation :")
print(x_validation_df)
print("="*40)
print("Testing Results")
print(testing_df.sort_values(by="R2(%)",ascending=False))
print("="*40)
print("Training Results")
print(training_df.sort_values(by="R2(%)",ascending=False))
print("\n")

# 7 HyperParameters Optimization

In [ ]:
param_grid = {
    "n_estimators":[100,200,300,500],
    "max_depth":[None,10,20,30],
    "min_sample_split":[2,5,10],
    "min_samples_leaf":[1,2,4],
    "max_features":["sqrt","log2",None]
}

random_search = RandomizedSearchCV(
    estimator=random_forest_model,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train,y_train)

print(random_search.best_params_)

print(-random_search.best_score_)

best_rf = random_search.best_estimator_

y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)